# 06 · Spark ile batch ve sensör akışı

AI4I CSV’sini toplu işliyor ve Kafka kayıtları üzerinde olay zamanı pencerelerini gösteriyoruz. Kafka kayıtları taşır ve saklar. Spark kayıtları okur, doğrular ve hesaplama yapar. PostgreSQL ayrı akış tablosunda doğrulanmış olayları saklar. [Türkçe rehber](../docs/spark_processing.md)

## 1. Yerel Spark oturumunu açma

Driver işleri planlar, executor görevleri yürütür. local[2] iki görev iş parçacığına izin verir ve ayrı bir cluster gerektirmez. Kafka connector oturum açılırken JVM’e eklenir. UTC ayarı olay zamanı pencerelerinin yerel saatten etkilenmesini önler.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import logging
import sys
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.database import connect
from streaming.spark_batch import SPARK_WORK, analyze, read_ai4i, spark_session
from streaming.spark_streaming import kafka_events, start_query

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
spark = spark_session(kafka=True)
print(spark.version, spark.sparkContext.master, spark.conf.get("spark.sql.session.timeZone"))

:: loading settings :: url = jar:file:/opt/pyspark-runtime/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /workspace/.spark/ivy/linux/cache
The jars for the packages stored in: /workspace/.spark/ivy/linux/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2cd784c-cf3e-445e-a944-235800df4491;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 in central


	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central


	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central


	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central


	found org.apache.commons#commons-pool2;2.12.1 in central


:: resolution report :: resolve 1269ms :: artifacts dl 285ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.12.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.4.2 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.4.2 from central in [default]
	org.apache.kafka#kafka-clients;3.9.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf     

26/11/25 10:26:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


4.1.2 local[2] UTC


## 2. Şema ve kayıt doğrulaması

Spark DataFrame sütun türlerini tanımlar. Sayıları açık şemayla okuyoruz ve 10.000 benzersiz UDI doğruluyoruz. Filtre gibi dönüşümler planı kurar. Count ve show gibi action’lar hesaplamayı başlatır. Buna lazy evaluation denir.

In [2]:
data = read_ai4i(spark)
data.printSchema()
print("Toplam kayıt", data.count())
print("Arızalı kayıt", data.where("machine_failure = 1").count())

root
 |-- udi: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- type_code: string (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- process_temperature: double (nullable = true)
 |-- rotational_speed: integer (nullable = true)
 |-- torque: double (nullable = true)
 |-- tool_wear: integer (nullable = true)
 |-- machine_failure: integer (nullable = true)
 |-- twf: integer (nullable = true)
 |-- hdf: integer (nullable = true)
 |-- pwf: integer (nullable = true)
 |-- osf: integer (nullable = true)
 |-- rnf: integer (nullable = true)



Toplam kayıt 10000


Arızalı kayıt 339


## 3. Spark SQL ile sonuçları yeniden üretme

Genel özeti, ürün tipi gruplarını, arıza durumuna göre sensör ortalamalarını ve ortalamanın üzerinde tork kaydedilen satırları hesaplıyoruz. PostgreSQL sorgusuyla toplam ve arıza sayısını karşılaştırıyoruz. Pandas tek Python işleminde hesaplar. Spark’ın JVM başlatma ve görev planlama maliyeti olduğundan küçük verilerde her zaman daha hızlı çalışmaz.

In [3]:
results = analyze(spark, data)
for name in ("overview", "product_rates", "sensor_means", "failure_modes"):
    display(pd.DataFrame([row.asDict() for row in results[name].collect()]))
print("Ortalamanın üzerinde tork", results["above_average"].count())

with connect() as connection:
    connection.execute("SET TRANSACTION READ ONLY")
    pg_total, pg_failures = connection.execute(
        "SELECT count(*), sum(machine_failure) FROM public.ml_training_data"
    ).fetchone()
assert (pg_total, pg_failures) == tuple(results["overview"].first())[:2]

,total_records,failure_count,normal_count,failure_rate_pct,avg_torque_nm,min_torque_nm,max_torque_nm
0,10000,339,9661,3.39,39.98691,3.8,76.6


,type_code,type_name,reading_count,labeled_count,failure_count,failure_rate_pct
0,H,High,1003,1003,21,2.09
1,L,Low,6000,6000,235,3.92
2,M,Medium,2997,2997,83,2.77


,machine_failure,record_count,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
0,0,9661,299.97,310.00,1540.26,39.63,106.69
1,1,339,300.89,310.29,1496.49,50.17,143.78


,failure_type,record_count
0,HDF,115
1,OSF,98
2,PWF,95
3,TWF,46
4,RNF,19


Ortalamanın üzerinde tork 5060


## 4. Structured Streaming ve olay zamanı

Kafka topic’inin mevcut kayıtlarını micro-batch sorgusuyla okuyoruz. Mesajlar açık JSON şemasıyla ayrıştırılır. Bir dakikalık UTC pencerelerinin olay sayısı ve ortalama torku güncellenir. İki dakikalık watermark eski pencere durumunun temizlenmesini sağlar. Bu pencere örneği ayrı geçici checkpoint kullanır ve sonuçları loglara yazar.

Örneğin en yeni olay 12.05 UTC’deyse watermark yaklaşık 12.03’e ilerler. 12.00 penceresine çok geç ulaşan bir olay daha sonra pencere hesabının dışında kalabilir.

In [4]:
with TemporaryDirectory(dir=SPARK_WORK) as checkpoint:
    events = kafka_events(spark)
    query = start_query(events, checkpoint, windows=True, available_now=True)
    try:
        assert query.awaitTermination(90)
        for row in query.recentProgress:
            print(dict(batch=row["batchId"], input_rows=row["numInputRows"],
                       event_time=row.get("eventTime", {})))
    finally:
        if query.isActive:
            query.stop()

INFO Callback Server Starting


INFO Socket listening on ('127.0.0.1', 42481)


26/11/25 10:27:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/11/25 10:27:12 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


INFO Python Server ready to receive messages


INFO Received command c on object id p0


INFO pencere batch=0 {'start': '2026-11-25T10:23:00Z', 'end': '2026-11-25T10:24:00Z', 'events': 100, 'avg_torque': 39.508, 'avg_air_temperature': 298.793}


INFO pencere batch=0 {'start': '2026-11-25T10:26:00Z', 'end': '2026-11-25T10:27:00Z', 'events': 200, 'avg_torque': 39.50800000000001, 'avg_air_temperature': 298.7930000000005}


INFO pencere batch=0 {'start': '2026-11-25T10:17:00Z', 'end': '2026-11-25T10:18:00Z', 'events': 100, 'avg_torque': 39.508, 'avg_air_temperature': 298.793}


INFO Received command c on object id p0


{'batch': 0, 'input_rows': 401, 'event_time': {'avg': '2026-11-25T10:23:23.060Z', 'max': '2026-11-25T10:26:19.410Z', 'min': '2026-11-25T10:17:15.566Z', 'watermark': '1970-01-01T00:00:00.000Z'}}
{'batch': 1, 'input_rows': 0, 'event_time': {'watermark': '2026-11-25T10:24:19.410Z'}}


## 5. PostgreSQL akış tablosu ve oturumu kapatma

Uçtan uca testin yazdığı ayrı tabloyu salt okunur sorguluyoruz. Normal akışta foreachBatch varsayılan olarak at-least-once çalışır. Checkpoint okuma konumunu saklar. UUID anahtarı aynı olay tekrar işlenirse ikinci satırın eklenmesini önler. Son adım Spark oturumunu kapatır.

In [5]:
try:
    with connect() as connection:
        connection.execute("SET TRANSACTION READ ONLY")
        total, unique = connection.execute(
            "SELECT count(*), count(DISTINCT event_id) FROM public.stream_sensor_readings"
        ).fetchone()
    print(f"Akış tablosu: {total} kayıt, {unique} benzersiz UUID")
    assert total == unique
finally:
    spark.stop()

Akış tablosu: 100 kayıt, 100 benzersiz UUID
